# Part 1 Data Cleaning and Validation

After analyzing all requirements, here is how I will structure the work:

Principles

- The raw table is never modified. Everything is read from ohlc_hw and written to new tables.
- Every raw row ends up in exactly one place: retained, removed (exact duplicate or all prices missing), or quarantined with a reason. 
- Repairs are flagged, never silent. Each repair gets a boolean flag column (e.g. flag_volume_filled, flag_ohlc_repaired, flag_adj_reconstructed, flag_extreme_return_kept).
- Original labels are preserved (raw_ticker), along with a ticker-normalization audit map.

Pipeline order (the order matters, since several steps depend on earlier ones)

- Load and normalize keys. Strip markup and whitespace and uppercase tickers, then parse ts to ISO YYYY-MM-DD. This happens before any grouping or deduplication.
- Profile (diagnostics only).Sort by (ticker, ts), and then
    - (low severity) exact dupli cate  
    - (high severity) conflicting key
    - (low severity) ticker format, ticker label that needed normalizing
    - ('medium' severity) Check missing cells per column, 
    - ('low' severity) missing trading dates versus the ohlc calendar (with a summary: "Ticker X is missing Y trading days between [start] and [end] dates"), 
    - ('medium' severity) impossible OHLC relationships, 
    - ('high' severity) non-positive prices, 
    - ('high' severity) flag extreme returns for manual checks, 
    - ('high' severity) flag zero volume days for manual checks.
- Generate a pandas dataframe 'Data Quality Report' based on the Profile (diagnostics only) step. The summary counts number of data issues per category. Column names are ticker | issue_type | issue_date | description | severity (high/medium/low).
- Deduplicate. Drop exact duplicates. For conflicting (ticker, ts) keys, resolve using evidence (the ohlc reference and OHLC validity), and quarantine the key if the evidence is inconclusive.
- Drop rows with all four raw prices missing, before any filling.
- - Sort by (ticker, ts), then repair raw prices. Invalid relationships become NaN, then forward-fill within ticker, then bounding with high = max(filled_high, open, close) and low = min(filled_low, open, close), with leading gaps quarantined. The OHLC bounds are rechecked afterwards, since a stale fill can violate today’s range. Leading gaps with nothing to carry forward are quarantined.
- Volume. Missing volume is filled with 0 and flagged.
- Extreme returns. Compute per-ticker z-scores as you specified based on repaired prices, then classify each hit:
    - Split-like: the volume jumps by a near-rational ratio
    - Data error: the move disagrees with the ohlc reference.
    - Inconclusive: quarantine and flag with a stated uncertainty.
- Only confirmed errors are interpolated, and only between valid neighbors.
- Missing Adjusted close. Take a date-specific factor adj_close / close from the matched ohlc row, then rebuild adj_close = clean_close × factor. There is no forward-filling of factors, and unresolved rows are quarantined and counted.
- Final validation. Check unique keys, finite positive prices, finite nonnegative volume, OHLC bounds, and adjusted-price consistency.
- Show before/after stats. 
- Write outputs in data.db: ohlc_hw_cleaned, also ohlc_hw_quarantine (with reason), ohlc_hw_removed (with reason) and ohlc_hw_ticker_audit

## 1.1 Load and Inspect the Data

In [1]:
# Load the `ohlc_hw` table from `data/data.db`

import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("data/data.db")

df = pd.read_sql("SELECT * from ohlc_hw", conn)
df_calendar = pd.read_sql("SELECT * from ohlc", conn)
conn.close()

display(df.head())
display(df_calendar.head())

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,ts,open,high,low,close,adj_close,volume,ticker
0,2024-10-14,161.40,162.14,160.90,161.60,161.60,5964000,JNJ
1,2024-10-11,162.18,162.66,161.20,161.46,161.46,5253500,JNJ
2,2024-10-10,161.36,161.39,160.00,160.51,160.51,4793900,JNJ
3,2024-09-30,161.57,162.18,160.68,162.06,162.06,6010407,JNJ
4,2024-09-27,162.00,163.43,161.34,161.40,161.40,7553905,JNJ


In [2]:
# Display basic statistics (`.info()`, `.describe()`)

df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


**<u>My Observation</u>**

*Hypothesis: only adj_close has missing values*

**Identify the columns and their meanings**

This table contains daily OHLC (Open, High, Low, Close) data for multiple stocks.
The columns are ts (observation date), ticker, raw open, high, low, close, vendor adj_close, and volume


In [3]:
# Show the date range and list of tickers
ts_col = pd.to_datetime(df['ts'])

#date range
print("date range in ohlc_hw")
ts_col.min(), ts_col.max()

date range in ohlc_hw


(Timestamp('1962-01-02 00:00:00'), Timestamp('2024-12-06 00:00:00'))

In [4]:
#compare with reference table ohlc
ts_col2 = pd.to_datetime(df_calendar['ts'])

#date range
print("date range in ohlc")
ts_col2.min(), ts_col2.max()

date range in ohlc


(Timestamp('1962-01-02 00:00:00'), Timestamp('2024-12-06 00:00:00'))

In [5]:
#list of tickers (sorted)
ticker_col = df['ticker']
sorted(ticker_col.unique())

['<a target="_blank"><span style="color: blue">AAPL</span></a>',
 '<a target="_blank"><span style="color: blue">AMGN</span></a>',
 '<a target="_blank"><span style="color: blue">AMZN</span></a>',
 '<a target="_blank"><span style="color: blue">AXP</span></a>',
 '<a target="_blank"><span style="color: blue">BA</span></a>',
 '<a target="_blank"><span style="color: blue">CAT</span></a>',
 '<a target="_blank"><span style="color: blue">CRM</span></a>',
 '<a target="_blank"><span style="color: blue">CSCO</span></a>',
 '<a target="_blank"><span style="color: blue">CVX</span></a>',
 '<a target="_blank"><span style="color: blue">DIS</span></a>',
 '<a target="_blank"><span style="color: blue">GS</span></a>',
 '<a target="_blank"><span style="color: blue">HD</span></a>',
 '<a target="_blank"><span style="color: blue">HON</span></a>',
 '<a target="_blank"><span style="color: blue">IBM</span></a>',
 '<a target="_blank"><span style="color: blue">JNJ</span></a>',
 '<a target="_blank"><span style="color

In [6]:
# Load and normalize keys. Strip markup and whitespace and uppercase tickers, then parse ts to ISO YYYY-MM-DD.

def normalize_keys(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Normalize the (ticker, ts) key columns before any grouping or dedup.

    * ticker: remove HTML tags (e.g. '<a ...><span ...>SHW</span></a>' -> 'SHW'),
      strip whitespace, upper-case.
    * ts: parse to a date and re-emit as ISO 'YYYY-MM-DD' strings so SQL date
      filters and key comparisons behave consistently.
    """
    missing = {"ticker", "ts"} - set(raw.columns)
    if missing:
        raise KeyError(f"raw data is missing required columns: {sorted(missing)}")

    df = raw.copy()
    df["raw_ticker"], df["raw_ts"] = df["ticker"], df["ts"]

    df["ticker"] = (
        df["raw_ticker"].astype("string")
        .str.replace(r"<[^>]+>", "", regex=True)  # drop every HTML tag, keep inner text
        .str.strip()
        .str.upper()
        .mask(lambda s: s.eq(""))                 # label that was only markup -> missing
    )

    # errors="coerce" turns unparseable dates into NaT instead of aborting the load.
    df["ts"] = pd.to_datetime(df["raw_ts"], errors="coerce", format="mixed").dt.strftime("%Y-%m-%d")

    # The quarantine step reads this column to explain why a key is unusable.
    problems = pd.DataFrame({
        "unusable ticker": df["ticker"].isna(),
        "unparseable ts": df["ts"].isna(),
    })
    df["key_issue"] = problems.apply(lambda r: "; ".join(r.index[r]) or None, axis=1)

    ticker_audit = (
        df.groupby(["raw_ticker", "ticker"], dropna=False).size()
        .rename("n_rows").reset_index()
        .sort_values(["ticker", "raw_ticker"], na_position="last", ignore_index=True)
    )
    # A missing label counts as changed so it appears in the audit review.
    ticker_audit["was_changed"] = (
        ticker_audit["raw_ticker"].astype("string")
        .ne(ticker_audit["ticker"].astype("string"))
        .fillna(True)
        .astype(bool)
    )
    return df, ticker_audit

In [7]:
# before and after ticker normalization
print("Unique tickers before normailization:", df["ticker"].nunique())
df, ticker_audit = normalize_keys(df)
print("Unique tickers after normalization: ", df["ticker"].nunique())

Unique tickers before normailization: 150
Unique tickers after normalization:  30


In [8]:
# Missing in the raw columns vs. unusable after normalization
print("raw_ticker missing:", df["raw_ticker"].isna().sum())
print("raw_ts missing:    ", df["raw_ts"].isna().sum())
print("ticker missing after normalization:", df["ticker"].isna().sum())
print("ts missing after normalization:    ", df["ts"].isna().sum())

# Breakdown of rows with an unusable key, and the rows themselves
print(df["key_issue"].value_counts(dropna=False))
display(df[df["key_issue"].notna()].head())

raw_ticker missing: 0
raw_ts missing:     0
ticker missing after normalization: 0
ts missing after normalization:     0
key_issue
None    397711
Name: count, dtype: int64


,ts,open,high,low,close,adj_close,volume,ticker,raw_ticker,raw_ts,key_issue


## 1.2 Detect Data Quality Issues

In [9]:
KEY = ["ticker", "ts"]
PRICE_COLS = ["open", "high", "low", "close"]
VALUE_COLS = PRICE_COLS + ["adj_close", "volume"]
REPORT_COLS = ["ticker", "issue_type", "issue_date", "description", "severity"]

# Single source of truth for severities, so every check and the summary agree.
SEVERITY = {
    "exact_duplicate": "low",
    "conflicting_key": "high",
    "ticker_format": "low",
    "missing_value": "medium",
    "missing_date": "low",
    "impossible_ohlc": "medium",
    "non_positive_price": "high",
    "extreme_return": "high",
    "zero_volume": "high",
}

df = df.sort_values(KEY, kind="stable", ignore_index=True)

# Rows with an unusable key cannot be located in the report, so checks skip them;
# the quarantine step deals with them later.
usable = df[df["key_issue"].isna()]
# Exact duplicates are dropped for diagnostics only (df is untouched), so one
# repeated record is not counted once per copy by the value checks below.
diag = usable.drop_duplicates(subset=KEY + VALUE_COLS)

dq_issues = {}  # issue_type -> report rows, assembled into the report in the last cell


def empty_report() -> pd.DataFrame:
    return pd.DataFrame(columns=REPORT_COLS)


def make_issues(ticker, issue_date, issue_type, description) -> pd.DataFrame:
    """Build Data Quality Report rows; scalar arguments broadcast over the Series ones."""
    out = pd.DataFrame({"ticker": ticker})
    out["issue_type"] = issue_type
    out["issue_date"] = issue_date
    out["description"] = description
    out["severity"] = SEVERITY[issue_type]
    return out[REPORT_COLS].reset_index(drop=True)


print(f"{len(df)} rows | {len(df) - len(usable)} with unusable key | "
      f"{len(usable) - len(diag)} exact duplicates | {len(diag)} rows used for value checks")

397711 rows | 0 with unusable key | 29460 exact duplicates | 368251 rows used for value checks


In [10]:
def check_exact_duplicates(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag every extra copy of a row identical in key and all value columns."""
    # keep="first": the first occurrence is the record, later copies are the issue.
    dups = rows[rows.duplicated(KEY + VALUE_COLS, keep="first")]
    return make_issues(dups["ticker"], dups["ts"], "exact_duplicate",
                       "Identical to an earlier row (after ticker normalization)")


dq_issues["exact_duplicate"] = check_exact_duplicates(usable)
print(len(dq_issues["exact_duplicate"]), "exact duplicate rows")
display(dq_issues["exact_duplicate"].head())

29460 exact duplicate rows


,ticker,issue_type,issue_date,description,severity
0,AAPL,exact_duplicate,1981-01-02,Identical to an earlier row (after ticker norm...,low
1,AAPL,exact_duplicate,1981-01-05,Identical to an earlier row (after ticker norm...,low
2,AAPL,exact_duplicate,1981-02-10,Identical to an earlier row (after ticker norm...,low
3,AAPL,exact_duplicate,1981-02-27,Identical to an earlier row (after ticker norm...,low
4,AAPL,exact_duplicate,1981-03-05,Identical to an earlier row (after ticker norm...,low


In [11]:
def check_conflicting_keys(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag (ticker, ts) keys that still hold different values once exact duplicates are gone.

    * One issue per key, naming the columns that disagree, so manual review knows where to look.
    * `rows` must already be free of exact duplicates.
    """
    conflicts = rows[rows.duplicated(KEY, keep=False)]
    if conflicts.empty:
        return empty_report()
    by_key = conflicts.groupby(KEY)
    n_rows = by_key.size().rename("n_rows")
    disagree = by_key[VALUE_COLS].nunique(dropna=False).gt(1)
    cols = disagree.apply(lambda r: ", ".join(r.index[r]), axis=1).rename("cols")
    out = pd.concat([n_rows, cols], axis=1).reset_index()
    desc = out["n_rows"].astype(str) + " rows share this key but differ in: " + out["cols"]
    return make_issues(out["ticker"], out["ts"], "conflicting_key", desc)


dq_issues["conflicting_key"] = check_conflicting_keys(diag)
print(len(dq_issues["conflicting_key"]), "conflicting keys")
display(dq_issues["conflicting_key"].head())

0 conflicting keys


,ticker,issue_type,issue_date,description,severity


In [12]:
def check_ticker_format(audit: pd.DataFrame) -> pd.DataFrame:
    """One issue per raw ticker label that needed normalizing (from the audit map)."""
    changed = audit[audit["was_changed"]]
    desc = pd.Series(
        [f"Label {raw!r} ({n} rows) " + (f"normalized to {tkr!r}" if pd.notna(tkr) else "left no usable ticker")
         for raw, tkr, n in zip(changed["raw_ticker"], changed["ticker"], changed["n_rows"])],
        index=changed.index, dtype=object,
    )
    # A label-level issue has no single date, so issue_date stays empty.
    return make_issues(changed["ticker"], None, "ticker_format", desc)


dq_issues["ticker_format"] = check_ticker_format(ticker_audit)
print(len(dq_issues["ticker_format"]), "labels needed normalizing")
display(dq_issues["ticker_format"].head())

120 labels needed normalizing


,ticker,issue_type,issue_date,description,severity
0,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""color: ...",low
1,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""color: ...",low
2,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""font-we...",low
3,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""font-we...",low
4,AMGN,ticker_format,None,"Label '<a target=""_blank""><span style=""color: ...",low


In [13]:
def check_missing_cells(rows: pd.DataFrame) -> pd.DataFrame:
    """One issue per missing cell in the price, adj_close and volume columns."""
    long = rows.melt(id_vars=KEY, value_vars=VALUE_COLS, var_name="column")
    missing = long[long["value"].isna()]
    return make_issues(missing["ticker"], missing["ts"], "missing_value", "Missing " + missing["column"])


print("Missing cells per column:")
display(diag[VALUE_COLS].isna().sum().to_frame("n_missing"))  # tests the "only adj_close is missing" hypothesis

dq_issues["missing_value"] = check_missing_cells(diag)
print(len(dq_issues["missing_value"]), "missing cells")

Missing cells per column:


,n_missing
open,0
high,0
low,0
close,0
adj_close,18374
volume,0


18374 missing cells


In [14]:
def check_missing_dates(rows: pd.DataFrame, calendar_ts: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare each ticker's dates with the trading days in the ohlc reference table.

    * The window is the ticker's own first-to-last date in ohlc_hw, so a ticker that
      starts late or ends early is not charged for dates outside its history.
    * Returns (report rows, one summary row per ticker).
    """
    calendar = pd.Index(sorted(pd.to_datetime(calendar_ts).dt.strftime("%Y-%m-%d").dropna().unique()))
    frames = []
    for ticker, g in rows.groupby("ticker"):
        seen = pd.Index(g["ts"].unique())
        start, end = seen.min(), seen.max()
        in_window = calendar[(calendar >= start) & (calendar <= end)]
        missing = in_window.difference(seen)
        if len(missing):
            frames.append(pd.DataFrame({"ticker": ticker, "ts": missing, "start": start, "end": end}))
    if not frames:
        return empty_report(), pd.DataFrame(columns=["ticker", "n_missing", "start", "end"])

    gaps = pd.concat(frames, ignore_index=True)
    summary = (gaps.groupby("ticker")
               .agg(n_missing=("ts", "size"), start=("start", "first"), end=("end", "first"))
               .reset_index())
    report = make_issues(gaps["ticker"], gaps["ts"], "missing_date",
                         "Trading day in the ohlc calendar has no row in ohlc_hw")
    return report, summary


dq_issues["missing_date"], missing_date_summary = check_missing_dates(diag, df_calendar["ts"])

for r in missing_date_summary.itertuples():
    print(f"Ticker {r.ticker} is missing {r.n_missing} trading days between {r.start} and {r.end}")
print(len(dq_issues["missing_date"]), "missing ticker-dates in total")

Ticker AAPL is missing 6 trading days between 1980-12-12 and 2024-10-18
Ticker AXP is missing 25 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 50 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker CVX is missing 25 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 1 trading days between 1962-01-02 and 2024-12-06
Ticker HON is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 25 trading days between 1962-01-02 and 2024-12-06
Ticker JNJ is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker JPM is missing 26 trading days between 1969-03-05 and 2024-10-18
Ticker KO is missing 26 trading days between 1962-01-02 and 2024-12-06
Ticker MCD is missing 26 trading days between 1966-07-05 and 2024-10-18
Ticker MMM is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker MRK is missing 26 trading days between 1962-01-02 and 2024-1

In [15]:
def check_impossible_ohlc(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag rows that break the OHLC relationships.

    * high must be >= max(open, close) and >= low; low must be <= min(open, close).
    * A NaN cell never violates a rule (it is reported as missing_value instead).
    """
    # skipna: if open is missing, close alone sets the bound.
    body_top = rows[["open", "close"]].max(axis=1)
    body_bottom = rows[["open", "close"]].min(axis=1)
    rules = pd.DataFrame({
        "high < max(open, close)": rows["high"] < body_top,
        "low > min(open, close)": rows["low"] > body_bottom,
        "high < low": rows["high"] < rows["low"],
    })
    bad = rules.any(axis=1)
    if not bad.any():
        return empty_report()
    desc = rules[bad].apply(lambda r: "; ".join(r.index[r]), axis=1)  # which rules failed
    return make_issues(rows.loc[bad, "ticker"], rows.loc[bad, "ts"], "impossible_ohlc", desc)


dq_issues["impossible_ohlc"] = check_impossible_ohlc(diag)
print(len(dq_issues["impossible_ohlc"]), "rows with impossible OHLC relationships")
display(dq_issues["impossible_ohlc"].head())

5 rows with impossible OHLC relationships


,ticker,issue_type,issue_date,description,severity
0,CVX,impossible_ohlc,1962-01-02,"low > min(open, close)",medium
1,JNJ,impossible_ohlc,1962-01-02,"low > min(open, close)",medium
2,MCD,impossible_ohlc,1966-07-05,"low > min(open, close)",medium
3,MRK,impossible_ohlc,1962-01-02,"low > min(open, close)",medium
4,PG,impossible_ohlc,1962-01-02,"low > min(open, close)",medium


In [16]:
def check_non_positive_prices(rows: pd.DataFrame) -> pd.DataFrame:
    """One issue per open/high/low/close/adj_close cell that is <= 0 or infinite."""
    long = rows.melt(id_vars=KEY, value_vars=PRICE_COLS + ["adj_close"], var_name="column")
    bad = long[long["value"].le(0) | np.isinf(long["value"])]  # NaN is not flagged here
    desc = "Non-positive/non-finite " + bad["column"] + " = " + bad["value"].astype(str)
    return make_issues(bad["ticker"], bad["ts"], "non_positive_price", desc)


dq_issues["non_positive_price"] = check_non_positive_prices(diag)
print(len(dq_issues["non_positive_price"]), "non-positive price cells")
display(dq_issues["non_positive_price"].head())

5 non-positive price cells


,ticker,issue_type,issue_date,description,severity
0,CVX,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high
1,JNJ,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high
2,MCD,non_positive_price,1966-07-05,Non-positive/non-finite open = 0.0,high
3,MRK,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high
4,PG,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high


In [17]:
def check_extreme_returns(rows: pd.DataFrame, n_std: float = 3) -> pd.DataFrame:
    """Flag closes whose daily return is more than n_std std away from the ticker's mean return.

    * Rule: abs(ret - mean_ret) > n_std * std_ret per ticker (sample std, ddof=1).
    * Returns skip missing and non-positive closes (a return spans the gap) and
      unresolved conflicting keys, so one bad row cannot poison the chain.
    * Diagnostic only: the cleaning step recomputes this on repaired prices.
    """
    clean = ~rows.duplicated(KEY, keep=False) & rows["close"].gt(0)
    s = rows.loc[clean, KEY + ["close"]].sort_values(KEY)

    s["prev_ts"] = s.groupby("ticker")["ts"].shift()
    s["ret"] = s["close"] / s.groupby("ticker")["close"].shift() - 1
    s["dev"] = s["ret"] - s.groupby("ticker")["ret"].transform("mean")
    s["std"] = s.groupby("ticker")["ret"].transform("std")

    hits = s[s["dev"].abs() > n_std * s["std"]]  # NaN std (too few returns) never flags
    if hits.empty:
        return empty_report()
    desc = hits.apply(
        lambda r: f"close {r['prev_ts']} -> {r['ts']}: return {r['ret']:+.1%} "
                  f"({r['dev'] / r['std']:+.1f} std from ticker mean)", axis=1)
    return make_issues(hits["ticker"], hits["ts"], "extreme_return", desc)


dq_issues["extreme_return"] = check_extreme_returns(diag)
print(len(dq_issues["extreme_return"]), "extreme returns to review")
display(dq_issues["extreme_return"].head())

5056 extreme returns to review


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,close 1980-12-24 -> 1980-12-26: return +9.2% (...,high
1,AAPL,extreme_return,1981-04-02,close 1981-04-01 -> 1981-04-02: return +8.8% (...,high
2,AAPL,extreme_return,1981-09-25,close 1981-09-24 -> 1981-09-25: return -13.0% ...,high
3,AAPL,extreme_return,1982-03-19,close 1982-03-18 -> 1982-03-19: return +9.0% (...,high
4,AAPL,extreme_return,1982-11-22,close 1982-11-19 -> 1982-11-22: return -8.9% (...,high


In [18]:
def check_zero_volume(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag days with volume == 0. Missing volume is a separate issue (missing_value)."""
    zero = rows[rows["volume"].eq(0)]
    return make_issues(zero["ticker"], zero["ts"], "zero_volume", "Volume is zero: review manually")


dq_issues["zero_volume"] = check_zero_volume(diag)
print(len(dq_issues["zero_volume"]), "zero-volume days")
display(dq_issues["zero_volume"].head())

75 zero-volume days


,ticker,issue_type,issue_date,description,severity
0,AXP,zero_volume,1972-12-14,Volume is zero: review manually,high
1,HON,zero_volume,1982-09-23,Volume is zero: review manually,high
2,HON,zero_volume,1982-09-24,Volume is zero: review manually,high
3,IBM,zero_volume,1973-10-09,Volume is zero: review manually,high
4,IBM,zero_volume,1975-01-27,Volume is zero: review manually,high


In [19]:
# Data Quality Report and summary

SEVERITY_ORDER = ["high", "medium", "low"]

frames = [f for f in dq_issues.values() if not f.empty]
dq_report = pd.concat(frames, ignore_index=True)
dq_report["severity"] = pd.Categorical(dq_report["severity"], categories=SEVERITY_ORDER, ordered=True)
dq_report = dq_report.sort_values(["severity", "issue_type", "ticker", "issue_date"], ignore_index=True)

# Summary lists every check, including those with zero hits, so it is clear each one ran.
dq_summary = (
    pd.DataFrame({
        "issue_type": list(dq_issues),
        "severity": [SEVERITY[k] for k in dq_issues],
        "n_issues": [len(v) for v in dq_issues.values()],
    })
    .assign(severity=lambda d: pd.Categorical(d["severity"], categories=SEVERITY_ORDER, ordered=True))
    .sort_values(["severity", "n_issues"], ascending=[True, False], ignore_index=True)
)

display(dq_summary)
print("Total issues:", len(dq_report))
display(dq_report.head(20))

,issue_type,severity,n_issues
0,extreme_return,high,5056
1,zero_volume,high,75
2,non_positive_price,high,5
3,conflicting_key,high,0
4,missing_value,medium,18374
5,impossible_ohlc,medium,5
6,exact_duplicate,low,29460
7,missing_date,low,445
8,ticker_format,low,120


Total issues: 53540


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,close 1980-12-24 -> 1980-12-26: return +9.2% (...,high
1,AAPL,extreme_return,1981-04-02,close 1981-04-01 -> 1981-04-02: return +8.8% (...,high
2,AAPL,extreme_return,1981-09-25,close 1981-09-24 -> 1981-09-25: return -13.0% ...,high
3,AAPL,extreme_return,1982-03-19,close 1982-03-18 -> 1982-03-19: return +9.0% (...,high
4,AAPL,extreme_return,1982-11-22,close 1982-11-19 -> 1982-11-22: return -8.9% (...,high
5,AAPL,extreme_return,1982-11-30,close 1982-11-29 -> 1982-11-30: return +10.4% ...,high
6,AAPL,extreme_return,1983-01-20,close 1983-01-19 -> 1983-01-20: return +11.2% ...,high
7,AAPL,extreme_return,1983-04-20,close 1983-04-19 -> 1983-04-20: return +8.9% (...,high
8,AAPL,extreme_return,1983-07-26,close 1983-07-25 -> 1983-07-26: return -9.3% (...,high
9,AAPL,extreme_return,1983-08-31,close 1983-08-30 -> 1983-08-31: return +13.3% ...,high


In [20]:
print(list(df.columns))

['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker', 'raw_ticker', 'raw_ts', 'key_issue']


## 1.3 Clean the Data

In [21]:
## Deduplicate

REF_RTOL = 1e-3  # relative tolerance when comparing a price with the ohlc reference (tunable)

# Stable id so every raw row ends up in exactly one place: retained, removed or quarantined.
# NOTE: df was sorted in 1.2, so this id is the position in that stable sort. Load with
# "SELECT rowid AS raw_row_id, * FROM ohlc_hw" if you want the table's own rowid instead.
if "raw_row_id" not in df.columns:
    df.insert(0, "raw_row_id", np.arange(len(df)))
N_RAW = len(df)

LOG_COLS = ["raw_row_id", "raw_ticker", "raw_ts", "ticker", "ts"] + VALUE_COLS
removed_log, quarantine_log = [], []


def log_rows(log: list, rows: pd.DataFrame, reason: str, note=None) -> None:
    """Record rows leaving the pipeline together with the reason (and an optional note)."""
    if rows.empty:
        return
    out = rows[LOG_COLS].copy()
    out["reason"], out["note"] = reason, note
    log.append(out)


def collect(log: list) -> pd.DataFrame:
    return pd.concat(log, ignore_index=True) if log else pd.DataFrame(columns=LOG_COLS + ["reason", "note"])


def ohlc_violations(rows: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """Boolean masks of rows whose high / low break the OHLC relationships.

    * high is bad if high < max(open, close) or high < low; low is bad if
      low > min(open, close) or low > high. When high < low both are bad.
    * NaN never violates a rule (missing cells are handled separately).
    """
    top, bottom = rows[["open", "close"]].max(axis=1), rows[["open", "close"]].min(axis=1)
    high_bad = (rows["high"] < top) | (rows["high"] < rows["low"])
    low_bad = (rows["low"] > bottom) | (rows["low"] > rows["high"])
    return high_bad, low_bad


def bound_ohlc(prices: pd.DataFrame) -> pd.DataFrame:
    """Widen high/low so the row's own open and close always sit inside [low, high]."""
    out = prices.copy()
    out["high"] = prices[["high", "open", "close"]].max(axis=1)
    out["low"] = prices[["low", "open", "close"]].min(axis=1)
    return out


def assert_price_integrity(rows: pd.DataFrame, step: str) -> None:
    """Recheck after every repair: finite positive prices and valid OHLC bounds."""
    prices = rows[PRICE_COLS]
    assert np.isfinite(prices).all().all() and (prices > 0).all().all(), f"{step}: missing/non-positive price"
    high_bad, low_bad = ohlc_violations(rows)
    assert not (high_bad | low_bad).any(), f"{step}: OHLC bounds violated"


# Reference table: same normalization, one trusted row per (ticker, ts).
ref, _ = normalize_keys(df_calendar)
ref = ref[ref["key_issue"].isna()].drop_duplicates(KEY + VALUE_COLS)
ref = ref[~ref.duplicated(KEY, keep=False)]  # conflicting reference keys are not trustworthy evidence
ref_cols = PRICE_COLS + ["adj_close"]
ref = ref[KEY + ref_cols].rename(columns={c: f"ref_{c}" for c in ref_cols})
# Date-specific adjustment factor; defined only where both reference prices are finite and positive.
pair = ref[["ref_close", "ref_adj_close"]]
ref["ref_factor"] = (pair["ref_adj_close"] / pair["ref_close"]).where(np.isfinite(pair).all(axis=1) & (pair > 0).all(axis=1))


def resolve_conflicting_keys(rows: pd.DataFrame, ref: pd.DataFrame) -> tuple[list, list]:
    """Return (ids to remove, ids to quarantine) for (ticker, ts) keys holding different values.

    * Evidence 1: the row's open/high/low/close all match the ohlc reference.
    * Evidence 2: the row has all prices present and no OHLC violation.
    * Evidence 1 is tried first; a key is resolved only if exactly one row wins on that
      evidence, otherwise every row of the key is quarantined.
    """
    conf = rows[rows.duplicated(KEY, keep=False)].merge(ref, on=KEY, how="left")
    conf["matches_ref"] = np.column_stack(
        [np.isclose(conf[c], conf[f"ref_{c}"], rtol=REF_RTOL) for c in PRICE_COLS]).all(axis=1)
    high_bad, low_bad = ohlc_violations(conf)
    conf["is_valid"] = ~(high_bad | low_bad) & conf[PRICE_COLS].notna().all(axis=1)

    drop, quarantine = [], []
    for _, g in conf.groupby(KEY):
        winner = next((g.loc[g[c], "raw_row_id"].iloc[0] for c in ("matches_ref", "is_valid") if g[c].sum() == 1), None)
        if winner is None:
            quarantine += g["raw_row_id"].tolist()
        else:
            drop += [i for i in g["raw_row_id"] if i != winner]
    return drop, quarantine


work = df.copy()

# 1) Unusable keys cannot be deduplicated or located: quarantine with the reason from normalize_keys.
bad_key = work["key_issue"].notna()
log_rows(quarantine_log, work[bad_key], "unusable key", work.loc[bad_key, "key_issue"])
work = work[~bad_key]

# 2) Exact duplicates (same key and identical values, NaN == NaN): keep the first copy.
exact_dup = work.duplicated(KEY + VALUE_COLS, keep="first")
log_rows(removed_log, work[exact_dup], "exact duplicate")
work = work[~exact_dup]

# 3) Conflicting keys: resolve with evidence, quarantine when inconclusive.
drop_ids, quarantine_ids = resolve_conflicting_keys(work, ref)
log_rows(removed_log, work[work["raw_row_id"].isin(drop_ids)], "conflicting key: lost to evidence")
log_rows(quarantine_log, work[work["raw_row_id"].isin(quarantine_ids)], "conflicting key: inconclusive evidence")
work = work[~work["raw_row_id"].isin(drop_ids + quarantine_ids)]
assert not work.duplicated(KEY).any(), "(ticker, ts) is still not unique"

print(f"raw {N_RAW} | unusable key {bad_key.sum()} | exact duplicates {exact_dup.sum()} | "
      f"conflicts removed {len(drop_ids)}, quarantined {len(quarantine_ids)} | left {len(work)}")

raw 397711 | unusable key 0 | exact duplicates 29460 | conflicts removed 0, quarantined 0 | left 368251


In [22]:
## Drop rows with all four raw prices missing (before any filling)

# A row with no price at all carries nothing to repair; filling it would invent a whole bar.
all_missing = work[PRICE_COLS].isna().all(axis=1)
log_rows(removed_log, work[all_missing], "all raw prices missing")
work = work[~all_missing]
print(f"removed {all_missing.sum()} rows with no raw price | left {len(work)}")

removed 0 rows with no raw price | left 368251


In [23]:
## Repair raw prices

def repair_raw_prices(rows: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Repair open/high/low/close within each ticker and flag every change.

    * Non-positive / non-finite prices are treated like missing cells.
    * Impossible OHLC: high -> NaN and/or low -> NaN (open and close are never touched).
    * Forward-fill within ticker, then bound: high = max(high, open, close), low = min(low, open, close).
    * Rows still missing a price after the fill (leading gap) are returned separately for quarantine.
    """
    rows = rows.sort_values(KEY, kind="stable").copy()  # fill and shift need (ticker, ts) order

    prices = rows[PRICE_COLS].replace([np.inf, -np.inf], np.nan)
    prices = prices.where(prices > 0)  # non-positive -> NaN
    was_missing = prices.isna()

    high_bad, low_bad = ohlc_violations(prices)
    prices.loc[high_bad, "high"] = np.nan
    prices.loc[low_bad, "low"] = np.nan

    # groupby(ticker).ffill never borrows a value across tickers.
    filled = prices.groupby(rows["ticker"]).ffill()
    leading = filled.isna().any(axis=1)  # nothing earlier in the ticker to carry forward

    # Bound only rows without a gap: bounding a NaN high would silently invent max(open, close).
    bounded = filled.copy()
    bounded.loc[~leading] = bound_ohlc(filled.loc[~leading])
    # A stale fill can fall outside today's range; this catches rows the bound had to widen.
    bound_changed = (bounded[["high", "low"]] != filled[["high", "low"]]).any(axis=1) & ~leading

    rows[PRICE_COLS] = bounded
    rows["flag_price_filled"] = was_missing.any(axis=1)
    rows["flag_close_repaired"] = was_missing["close"]  # close is never changed by bounding; used for adj_close later
    rows["flag_ohlc_repaired"] = high_bad | low_bad | bound_changed

    gap_cols = [", ".join(c for c, m in zip(PRICE_COLS, flags) if m) for flags in filled[leading].isna().to_numpy()]
    leading_rows = rows[leading].assign(note=[f"no prior value for: {c}" for c in gap_cols])
    return rows[~leading], leading_rows


work, leading_rows = repair_raw_prices(work)
log_rows(quarantine_log, leading_rows, "leading gap", leading_rows["note"])
assert_price_integrity(work, "after raw-price repair")

print(f"quarantined {len(leading_rows)} leading-gap rows | left {len(work)}")
display(work[["flag_price_filled", "flag_close_repaired", "flag_ohlc_repaired"]].sum().to_frame("n_rows"))

quarantined 5 leading-gap rows | left 368246


,n_rows
flag_price_filled,0
flag_close_repaired,0
flag_ohlc_repaired,0


In [24]:
## Volume

# Unknown volume is not evidence of no trading, so the fill value (0) is always paired with a flag.
# Negative or infinite volume cannot be real either; it is handled the same way.
bad_volume = ~np.isfinite(work["volume"]) | work["volume"].lt(0)
work["flag_volume_filled"] = bad_volume
work.loc[bad_volume, "volume"] = 0.0
print(f"filled {bad_volume.sum()} volume cells with 0 (flagged)")

filled 0 volume cells with 0 (flagged)


In [25]:
## Extreme returns: classify each hit

N_STD = 3
# Volume ratios treated as split-like (2-for-1, 3-for-2, reverse splits ...) and their tolerance (tunable).
SPLIT_RATIOS = np.array([2, 3, 4, 5, 10, 1/2, 1/3, 1/4, 1/5, 1/10, 3/2, 2/3])
SPLIT_TOL = 0.02
KEPT_CLASSES = ["split_like", "ref_confirmed", "revert_of_data_error"]


def zscore_returns(rows: pd.DataFrame, n_std: float = N_STD) -> pd.DataFrame:
    """Daily close return per ticker and whether abs(ret - mean_ret) > n_std * std_ret.

    * Sample std (ddof=1), per ticker, on the sorted repaired closes.
    * A ticker with std 0 or fewer than 2 returns never flags.
    """
    s = rows.sort_values(KEY, kind="stable").reset_index(drop=True)
    by_ticker = s.groupby("ticker")
    s["prev_close"] = by_ticker["close"].shift()
    s["prev_volume"] = by_ticker["volume"].shift()
    s["prev_vol_filled"] = by_ticker["flag_volume_filled"].shift(fill_value=True)
    s["ret"] = s["close"] / s["prev_close"] - 1
    stats = s.groupby("ticker")["ret"]
    # |ret - mean| > n * std  <=>  |z| > n
    s["zscore"] = (s["ret"] - stats.transform("mean")) / stats.transform("std")
    s["is_hit"] = s["zscore"].abs() > n_std
    return s


def classify_extreme_returns(s: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    """Label each hit with the evidence available.

    * data_error: close disagrees with the ohlc reference (checked first; the reference wins over volume).
    * split_like: volume jumps by a near-rational ratio (needs real volume on both days).
    * ref_confirmed: close agrees with the reference, so the move is real news: keep and flag.
    * revert_of_data_error: otherwise-inconclusive hit right after a data error (it is the reversion).
    * inconclusive: no evidence either way -> quarantined.
    """
    s = s.merge(ref[KEY + ["ref_close"]], on=KEY, how="left")  # left merge keeps the sorted order
    ref_agrees = pd.Series(np.isclose(s["close"], s["ref_close"], rtol=REF_RTOL), index=s.index)  # False if no ref
    ref_disagrees = s["ref_close"].notna() & ~ref_agrees

    # Volume ratio is undefined for zero, missing or filled volume: such hits cannot be called splits.
    vol_known = (s["volume"] > 0) & (s["prev_volume"] > 0) & ~s["flag_volume_filled"] & ~s["prev_vol_filled"]
    s["vol_ratio"] = (s["volume"] / s["prev_volume"]).where(vol_known)
    s["close_ratio"] = 1 + s["ret"]  # evidence only: a split should show close_ratio ~ 1 / vol_ratio
    s["close_vs_ref"] = s["close"] / s["ref_close"] - 1
    is_split = np.abs(s["vol_ratio"].to_numpy()[:, None] / SPLIT_RATIOS - 1).min(axis=1) <= SPLIT_TOL

    outcome = np.select([ref_disagrees, is_split, ref_agrees],
                        ["data_error", "split_like", "ref_confirmed"], default="inconclusive")
    s["er_class"] = np.where(s["is_hit"], outcome, None)

    prev_class = s.groupby("ticker")["er_class"].shift()
    s.loc[s["er_class"].eq("inconclusive") & prev_class.eq("data_error"), "er_class"] = "revert_of_data_error"

    # Stated uncertainty for the quarantine table.
    vr = s["vol_ratio"].round(3).astype(str)
    s["note"] = np.where(s["vol_ratio"].isna(),
                         "No ohlc reference row; volume ratio undefined (zero, missing or filled volume)",
                         "No ohlc reference row; volume ratio " + vr + " is not a near-rational split ratio")
    return s


er = classify_extreme_returns(zscore_returns(work), ref)
er_hits = er[er["is_hit"]]
display(er_hits["er_class"].value_counts().to_frame("n_hits"))
display(er_hits[["ticker", "ts", "ret", "zscore", "vol_ratio", "close_ratio", "close_vs_ref", "er_class"]].head(10))

# Attach the class to the working table; the z-scores above were computed before any row is dropped.
by_id = er_hits.set_index("raw_row_id")
work["extreme_return_class"] = work["raw_row_id"].map(by_id["er_class"])
work["flag_extreme_return_kept"] = work["extreme_return_class"].isin(KEPT_CLASSES)

# Inconclusive: quarantine with the stated uncertainty.
inconclusive = work["extreme_return_class"].eq("inconclusive")
log_rows(quarantine_log, work[inconclusive], "inconclusive extreme return", work.loc[inconclusive, "raw_row_id"].map(by_id["note"]))
work = work[~inconclusive]
print(f"quarantined {inconclusive.sum()} inconclusive rows | left {len(work)}")

,n_hits
er_class,
ref_confirmed,4637
split_like,420


,ticker,ts,ret,zscore,vol_ratio,close_ratio,close_vs_ref,er_class
9,AAPL,1980-12-26,0.092288,3.275625,1.157723,1.092288,0.0,ref_confirmed
76,AAPL,1981-04-02,0.087659,3.109418,0.921762,1.087659,0.0,ref_confirmed
197,AAPL,1981-09-25,-0.129776,-4.699052,1.891065,0.870224,0.0,ref_confirmed
318,AAPL,1982-03-19,0.090173,3.199699,1.168191,1.090173,0.0,ref_confirmed
490,AAPL,1982-11-22,-0.089089,-3.237912,1.040516,0.910911,0.0,ref_confirmed
495,AAPL,1982-11-30,0.103871,3.691603,3.186996,1.103871,0.0,ref_confirmed
531,AAPL,1983-01-20,0.111518,3.966231,4.172168,1.111518,0.0,ref_confirmed
593,AAPL,1983-04-20,0.088684,3.146230,1.232832,1.088684,0.0,ref_confirmed
660,AAPL,1983-07-26,-0.092718,-3.368230,3.519344,0.907282,0.0,ref_confirmed
686,AAPL,1983-08-31,0.133074,4.740351,0.855898,1.133074,0.0,ref_confirmed


quarantined 0 inconclusive rows | left 368246


In [26]:
## Interpolate confirmed data errors (between valid neighbors only)

def interpolate_data_errors(rows: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Replace the prices of confirmed data-error rows by linear interpolation within ticker.

    * All four prices of the row are rebuilt (a bad close makes the rest of the bar unreliable),
      then re-bounded so open/close sit inside [low, high].
    * limit_area="inside": never extrapolate. Errors without a valid neighbor on both sides
      are returned separately for quarantine.
    * Only error rows are NaN here, so no other cell can be interpolated by accident.
    """
    rows = rows.sort_values(KEY, kind="stable").copy()
    is_error = rows["extreme_return_class"].eq("data_error")

    prices = rows[PRICE_COLS].copy()
    prices.loc[is_error] = np.nan
    interp = prices.groupby(rows["ticker"]).transform(lambda s: s.interpolate(limit_area="inside"))  # linear by position

    unresolved = is_error & interp.isna().any(axis=1)
    fixed = is_error & ~unresolved

    bounded = bound_ohlc(interp.loc[fixed])
    bound_changed = (bounded[["high", "low"]] != interp.loc[fixed, ["high", "low"]]).any(axis=1)

    rows.loc[fixed, PRICE_COLS] = bounded
    rows["flag_extreme_return_interpolated"] = fixed
    rows["flag_close_repaired"] |= fixed          # adj_close must be rebuilt for these rows
    rows.loc[fixed[fixed].index[bound_changed], "flag_ohlc_repaired"] = True
    return rows[~unresolved], rows[unresolved]


work, unresolved_errors = interpolate_data_errors(work)
log_rows(quarantine_log, unresolved_errors, "confirmed data error without valid neighbors",
         "close disagrees with ohlc reference; cannot interpolate without a valid neighbor on both sides")
assert_price_integrity(work, "after interpolation")

print(f"interpolated {work['flag_extreme_return_interpolated'].sum()} rows | "
      f"quarantined {len(unresolved_errors)} | left {len(work)}")
# Recheck on the repaired series (statistics change once the errors are gone).
print("extreme returns remaining on repaired closes:", zscore_returns(work)["is_hit"].sum())

interpolated 0 rows | quarantined 0 | left 368246
extreme returns remaining on repaired closes: 5057


In [27]:
## Missing adjusted close

ADJ_RTOL = 1e-3  # tolerance for the consistency diagnostic below

# Convention: adj_close = clean_close * (reference adj_close / reference close) on the SAME date.
# The factor is never carried across dates, so a corporate action cannot leak into other days.
work = work.merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")

adj_invalid = ~np.isfinite(work["adj_close"]) | work["adj_close"].le(0)
# A vendor adj_close no longer matches a close that was filled or interpolated, so rebuild those too.
rebuild = adj_invalid | work["flag_close_repaired"]
has_factor = work["ref_factor"].notna()

unresolved = rebuild & ~has_factor
note = np.where(adj_invalid, "adj_close missing/invalid and no trusted reference factor",
                "close was repaired and no trusted reference factor")
log_rows(quarantine_log, work[unresolved], "adj_close unresolved", pd.Series(note, index=work.index)[unresolved])

work["flag_adj_reconstructed"] = rebuild & has_factor
work.loc[work["flag_adj_reconstructed"], "adj_close"] = work["close"] * work["ref_factor"]
work = work[~unresolved]

# Consistency diagnostic on untouched vendor rows: does adj_close / close match the reference factor?
vendor = work[~work["flag_adj_reconstructed"] & work["ref_factor"].notna()]
mismatch = ~np.isclose(vendor["adj_close"] / vendor["close"], vendor["ref_factor"], rtol=ADJ_RTOL)
print(f"adj_close rebuilt {work['flag_adj_reconstructed'].sum()} | quarantined {unresolved.sum()} | "
      f"vendor rows disagreeing with reference factor: {mismatch.sum()} of {len(vendor)} (reported, not altered)")

work = work.drop(columns="ref_factor").reset_index(drop=True)
assert_price_integrity(work, "after adj_close rebuild")
assert np.isfinite(work["adj_close"]).all() and (work["adj_close"] > 0).all()


adj_close rebuilt 18374 | quarantined 0 | vendor rows disagreeing with reference factor: 10417 of 349872 (reported, not altered)


In [28]:
# Row accounting: every raw row is in exactly one place.
removed, quarantined = collect(removed_log), collect(quarantine_log)
ids = pd.concat([work["raw_row_id"], removed["raw_row_id"], quarantined["raw_row_id"]])
assert ids.is_unique and len(ids) == N_RAW, "row accounting does not reconcile"
print(f"raw {N_RAW} = retained {len(work)} + removed {len(removed)} + quarantined {len(quarantined)}")
display(removed["reason"].value_counts().to_frame("removed"))
display(quarantined["reason"].value_counts().to_frame("quarantined"))

raw 397711 = retained 368246 + removed 29460 + quarantined 5


,removed
reason,
exact duplicate,29460


,quarantined
reason,
leading gap,5


In [29]:
## Final validation

ISO_DATE = r"\d{4}-\d{2}-\d{2}"
FLAG_COLS = [c for c in work.columns if c.startswith("flag_")]


def run_final_checks(clean, raw, removed, quarantined, ref) -> pd.DataFrame:
    """Independent recheck of the cleaned table: one row per check with its failure count.

    * Bounds are rewritten explicitly instead of reusing `ohlc_violations`, so a bug in
      the repair helpers cannot hide itself.
    * "No silent repairs": a row with no repair flag must equal its raw row.
    * Adjusted-price consistency: adj_close / close is finite and positive on every row, and
      every rebuilt adj_close equals close x the SAME-DATE reference factor.
    """
    prices = clean[PRICE_COLS + ["adj_close"]]
    o, h, l, c = (clean[k] for k in PRICE_COLS)
    factor = clean["adj_close"] / c
    ticker = clean["ticker"].astype(str)

    # Same-date reference factor for each retained row (ref has one row per key, order is kept).
    ref_factor = pd.Series(
        clean[KEY].merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")["ref_factor"].to_numpy(),
        index=clean.index)
    factor_matches = pd.Series(np.isclose(factor, ref_factor, rtol=1e-9), index=clean.index)  # False if no ref

    # Raw values of the same rows; NaN != NaN is True, so a raw NaN left unflagged is caught too.
    cmp = clean.merge(raw[["raw_row_id"] + VALUE_COLS], on="raw_row_id", suffixes=("", "_raw"))
    changed = lambda cols: pd.DataFrame({k: cmp[k].ne(cmp[f"{k}_raw"]) for k in cols}).any(axis=1)
    price_flagged = cmp[["flag_price_filled", "flag_ohlc_repaired", "flag_extreme_return_interpolated"]].any(axis=1)

    ids = pd.concat([clean["raw_row_id"], removed["raw_row_id"], quarantined["raw_row_id"]])

    checks = {
        "unique (ticker, ts) keys": clean.duplicated(KEY).sum(),
        "ticker label clean (no markup/whitespace, upper-case)":
            (ticker.str.contains(r"[<>\s]") | ticker.ne(ticker.str.upper())).sum(),
        "ts is an ISO date YYYY-MM-DD":
            (~clean["ts"].astype(str).str.fullmatch(ISO_DATE)
             | pd.to_datetime(clean["ts"], format="%Y-%m-%d", errors="coerce").isna()).sum(),
        "no missing / non-finite / non-positive raw or adjusted price": (~np.isfinite(prices) | prices.le(0)).sum().sum(),
        "finite non-negative volume": (~np.isfinite(clean["volume"]) | clean["volume"].lt(0)).sum(),
        "OHLC bounds: low <= open, close <= high": ((h < o) | (h < c) | (h < l) | (l > o) | (l > c)).sum(),
        "adj_close / close finite and positive": (~np.isfinite(factor) | factor.le(0)).sum(),
        "rebuilt adj_close = close x same-date reference factor":
            (clean["flag_adj_reconstructed"] & ~factor_matches).sum(),
        "every repaired close has a rebuilt adj_close":
            (clean["flag_close_repaired"] & ~clean["flag_adj_reconstructed"]).sum(),
        "no silent repairs: open/high/low/close": (changed(PRICE_COLS) & ~price_flagged).sum(),
        "no silent repairs: adj_close": (changed(["adj_close"]) & ~cmp["flag_adj_reconstructed"]).sum(),
        "no silent repairs: volume": (changed(["volume"]) & ~cmp["flag_volume_filled"]).sum(),
        "row accounting: every raw row in exactly one table":
            ids.duplicated().sum() + abs(len(ids) - len(raw)) + (~ids.isin(raw["raw_row_id"])).sum(),
        "removed / quarantined rows all carry a reason":
            removed["reason"].isna().sum() + quarantined["reason"].isna().sum(),
    }
    out = pd.DataFrame({"check": list(checks), "n_failures": [int(v) for v in checks.values()]})
    return out.assign(passed=out["n_failures"].eq(0))


removed, quarantined = collect(removed_log), collect(quarantine_log)
final_checks = run_final_checks(work, df, removed, quarantined, ref)
display(final_checks)
assert final_checks["passed"].all(), "final validation failed: see the table above"

# Informational, not a failure: untouched vendor adj_close values that disagree with the reference
# factor on the same date. They were reported (not altered) in the adj_close step.
vendor = work.merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")
vendor = vendor[~vendor["flag_adj_reconstructed"] & vendor["ref_factor"].notna()]
n_disagree = (~np.isclose(vendor["adj_close"] / vendor["close"], vendor["ref_factor"], rtol=ADJ_RTOL)).sum()
print(f"All {len(final_checks)} checks passed on {len(work)} rows. "
      f"Vendor adj_close disagreeing with the reference factor: {n_disagree} of {len(vendor)} rows.")

,check,n_failures,passed
0,"unique (ticker, ts) keys",0,True
1,"ticker label clean (no markup/whitespace, uppe...",0,True
2,ts is an ISO date YYYY-MM-DD,0,True
3,no missing / non-finite / non-positive raw or ...,0,True
4,finite non-negative volume,0,True
5,"OHLC bounds: low <= open, close <= high",0,True
6,adj_close / close finite and positive,0,True
7,rebuilt adj_close = close x same-date referenc...,0,True
8,every repaired close has a rebuilt adj_close,0,True
9,no silent repairs: open/high/low/close,0,True


All 14 checks passed on 368246 rows. Vendor adj_close disagreeing with the reference factor: 10417 of 349872 rows.


In [30]:
## Before / after statistics

def profile_stats(rows: pd.DataFrame) -> pd.Series:
    """Headline data quality metrics for one version of the table.

    * Row and ticker counts use every row. All other counts use distinct rows with a usable
      key, the same basis as the Data Quality Report in 1.2, so 'before' ties back to it.
    """
    distinct = rows[rows["key_issue"].isna()].drop_duplicates(KEY + VALUE_COLS)
    cells = distinct[PRICE_COLS + ["adj_close"]]
    high_bad, low_bad = ohlc_violations(distinct)
    stats = {
        "rows": len(rows),
        "unique tickers (normalized)": rows["ticker"].nunique(),
        "exact duplicate rows": rows.duplicated(KEY + VALUE_COLS).sum(),
        "extra rows on a repeated (ticker, ts) key": distinct.duplicated(KEY).sum(),
        **{f"missing {c}": distinct[c].isna().sum() for c in VALUE_COLS},
        "non-positive / infinite price cells": (cells.le(0) | np.isinf(cells)).sum().sum(),
        "impossible OHLC rows": (high_bad | low_bad).sum(),
        "zero-volume rows": distinct["volume"].eq(0).sum(),
        "extreme-return days (3 std)": len(check_extreme_returns(distinct)),
    }
    return pd.Series(stats).astype(int)


print(f"Ticker labels: {df['raw_ticker'].nunique()} raw -> {work['ticker'].nunique()} normalized (retained rows)")

before_after = pd.concat({"before": profile_stats(df), "after": profile_stats(work)}, axis=1)
before_after["change"] = before_after["after"] - before_after["before"]
display(before_after)
# 'after' still shows extreme-return days: moves confirmed by the reference or split-like are kept
# and flagged, and the z-scores are recomputed on the repaired series.

# Where every raw row went.
accounting = pd.concat([
    pd.DataFrame({"table": "retained (ohlc_hw_cleaned)", "reason": "-", "n_rows": [len(work)]}),
    removed.groupby("reason").size().rename("n_rows").reset_index().assign(table="removed (ohlc_hw_removed)"),
    quarantined.groupby("reason").size().rename("n_rows").reset_index().assign(table="quarantined (ohlc_hw_quarantine)"),
], ignore_index=True)[["table", "reason", "n_rows"]]
accounting["pct_of_raw"] = (accounting["n_rows"] / N_RAW * 100).round(2)
display(accounting)
print(f"raw {N_RAW} = retained {len(work)} + removed {len(removed)} + quarantined {len(quarantined)}")

# How much of the retained data was repaired, and how the kept extreme returns were classified.
flag_summary = work[FLAG_COLS].sum().to_frame("n_rows")
flag_summary["pct_of_retained"] = (flag_summary["n_rows"] / len(work) * 100).round(2)
display(flag_summary)
display(work["extreme_return_class"].value_counts(dropna=False).to_frame("n_rows"))

# Quarantine concentrated in a few tickers usually points at a vendor problem worth reviewing.
display(quarantined.groupby("ticker").size().sort_values(ascending=False).head(10).to_frame("n_quarantined"))

# Distribution shift: repairs should not move the overall shape of the data.
display(pd.concat({"before": df[VALUE_COLS].describe().T, "after": work[VALUE_COLS].describe().T}, axis=1).round(2))

Ticker labels: 150 raw -> 30 normalized (retained rows)


,before,after,change
rows,397711,368246,-29465
unique tickers (normalized),30,30,0
exact duplicate rows,29460,0,-29460
"extra rows on a repeated (ticker, ts) key",0,0,0
missing open,0,0,0
missing high,0,0,0
missing low,0,0,0
missing close,0,0,0
missing adj_close,18374,0,-18374
missing volume,0,0,0


,table,reason,n_rows,pct_of_raw
0,retained (ohlc_hw_cleaned),-,368246,92.59
1,removed (ohlc_hw_removed),exact duplicate,29460,7.41
2,quarantined (ohlc_hw_quarantine),leading gap,5,0.00


raw 397711 = retained 368246 + removed 29460 + quarantined 5


,n_rows,pct_of_retained
flag_price_filled,0,0.00
flag_close_repaired,0,0.00
flag_ohlc_repaired,0,0.00
flag_volume_filled,0,0.00
flag_extreme_return_kept,5057,1.37
flag_extreme_return_interpolated,0,0.00
flag_adj_reconstructed,18374,4.99


,n_rows
extreme_return_class,
NaN,363189
ref_confirmed,4637
split_like,420


,n_quarantined
ticker,
CVX,1
JNJ,1
MCD,1
MRK,1
PG,1


before                                                           \
              count         mean           std   min         25%         50%   
open       397711.0        46.29  6.713000e+01  0.00        4.21       19.85   
high       397711.0        46.75  6.778000e+01  0.01        4.24       20.06   
low        397711.0        45.82  6.648000e+01  0.00        4.16       19.63   
close      397711.0        46.29  6.714000e+01  0.01        4.20       19.86   
adj_close  377856.0        51.65  1.945800e+02  0.00        1.39       11.96   
volume     397711.0  30977733.69  1.266237e+08  0.00  1730700.00  4648200.00   

                                         after                             \
                   75%           max     count         mean           std   
open             57.53  6.042600e+02  368246.0        46.31  6.714000e+01   
high             58.13  6.086300e+02  368246.0        46.77  6.779000e+01   
low              56.95  5.978800e+02  368246.0        45.84  6.649000e+01   
close            57.53  6.054000e+02  368246.0        46.32  6.715000e+01   
adj_close        43.77  1.044251e+04  368246.0        51.00  1.906300e+02   
volume     11023796.50  9.276606e+09  368246.0  30951838.94  1.266153e+08   

                                                                    
            min         25%         50%          75%           max  
open       0.01        4.21       19.89        57.56  6.042600e+02  
high       0.01        4.25       20.09        58.16  6.086300e+02  
low        0.00        4.17       19.67        57.00  5.978800e+02  
close      0.01        4.21       19.91        57.57  6.054000e+02  
adj_close  0.00        1.39       11.96        43.70  1.044251e+04  
volume     0.00  1731514.50  4649648.00  11021601.50  9.276606e+09

In [31]:
## Write outputs to data.db

from contextlib import closing

DB_PATH = "data/data.db"
CLEAN_COLS = (["raw_row_id", "ticker", "ts"] + VALUE_COLS + ["raw_ticker", "raw_ts"]
              + FLAG_COLS + ["extreme_return_class"])


def for_sqlite(frame: pd.DataFrame) -> pd.DataFrame:
    """Convert to plain dtypes SQLite stores predictably.

    * bool -> 0/1, timestamps -> ISO date text, pandas string -> object, missing -> NULL.
    """
    out = frame.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_bool_dtype(s):
            out[col] = s.astype(int)
        elif pd.api.types.is_datetime64_any_dtype(s):
            out[col] = s.dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_string_dtype(s):
            out[col] = s.astype(object).where(s.notna(), None)
    return out


outputs = {
    "ohlc_hw_cleaned": work[CLEAN_COLS],
    "ohlc_hw_quarantine": quarantined,   # LOG_COLS + reason + note (stated uncertainty)
    "ohlc_hw_removed": removed,           # LOG_COLS + reason
    "ohlc_hw_ticker_audit": ticker_audit,
    "ohlc_hw_dq_report": dq_report.assign(severity=dq_report["severity"].astype(str)),
}
# Guard: the raw table and the reference table are never written to.
assert not {"ohlc_hw", "ohlc"} & set(outputs)

# One transaction: either every output table is replaced or none is.
with closing(sqlite3.connect(DB_PATH)) as conn, conn:
    for name, frame in outputs.items():
        for_sqlite(frame).to_sql(name, conn, if_exists="replace", index=False)

# Read back from the database, not from memory, to verify what was actually saved.
with closing(sqlite3.connect(DB_PATH)) as conn:
    counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in [*outputs, "ohlc_hw"]}
    n_bad_ts = conn.execute(
        "SELECT COUNT(*) FROM ohlc_hw_cleaned "
        "WHERE ts NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'").fetchone()[0]
    n_dup_keys = conn.execute(
        "SELECT COUNT(*) FROM (SELECT 1 FROM ohlc_hw_cleaned GROUP BY ticker, ts HAVING COUNT(*) > 1)").fetchone()[0]
    n_bad_rows = conn.execute("""
        SELECT COUNT(*) FROM ohlc_hw_cleaned
        WHERE open IS NULL OR high IS NULL OR low IS NULL OR close IS NULL OR adj_close IS NULL OR volume IS NULL
           OR open <= 0 OR high <= 0 OR low <= 0 OR close <= 0 OR adj_close <= 0 OR volume < 0
           OR high < MAX(open, close, low) OR low > MIN(open, close, high)""").fetchone()[0]
    ts_range = conn.execute("SELECT MIN(ts), MAX(ts) FROM ohlc_hw_cleaned").fetchone()

display(pd.Series(counts, name="rows in database").to_frame())
assert counts["ohlc_hw_cleaned"] == len(work)
assert counts["ohlc_hw_removed"] == len(removed) and counts["ohlc_hw_quarantine"] == len(quarantined)
assert counts["ohlc_hw_cleaned"] + counts["ohlc_hw_removed"] + counts["ohlc_hw_quarantine"] == N_RAW
assert counts["ohlc_hw"] == N_RAW, "raw table row count changed"
assert n_bad_ts == 0 and n_dup_keys == 0 and n_bad_rows == 0
print(f"Saved and verified. ts range {ts_range[0]} to {ts_range[1]}; "
      f"{counts['ohlc_hw_cleaned']} cleaned + {counts['ohlc_hw_removed']} removed + "
      f"{counts['ohlc_hw_quarantine']} quarantined = {N_RAW} raw rows.")

,rows in database
ohlc_hw_cleaned,368246
ohlc_hw_quarantine,5
ohlc_hw_removed,29460
ohlc_hw_ticker_audit,150
ohlc_hw_dq_report,53540
ohlc_hw,397711


Saved and verified. ts range 1962-01-02 to 2024-12-06; 368246 cleaned + 29460 removed + 5 quarantined = 397711 raw rows.
